# Modelo de medición con covariables (comentario 23)

Responde el comentario 23 del profesor guía e incorpora al modelo de medición las covariables construidas en el notebook 07. La pregunta es si la conclusión sobre **H3** sobrevive a una medida distinta del nivel socioeconómico y a controles territoriales.

**Requiere** los notebooks 02, 06 y 07 ejecutados (`dataset_analitico_mate_covariables.csv`).

### Por qué tres especificaciones y no una

El IVE y el GSE correlacionan −0,87: miden casi lo mismo. Si entran juntos, el modelo no puede separar sus efectos y el coeficiente del GSE pasa a significar "GSE a igual IVE", una cantidad sin sentido sustantivo y con intervalos enormes. Por eso se usan como **medidas alternativas** del nivel socioeconómico, no simultáneas:

| Especificación | Medida socioeconómica | Controles |
|---|---|---|
| A | GSE (cinco grupos) | Nivel previo |
| B | GSE | Nivel previo + territorio |
| C | IVE (continuo), en lugar del GSE | Nivel previo + territorio |

Controles territoriales: pobreza comunal previa, densidad comunal, ruralidad, matrícula y dependencia.

**Lectura de H3.** En A y B se contrasta con la diferencia de pendiente entre GSE Alto y Bajo (H3 predice positiva). En C, con el coeficiente del IVE sobre la pendiente (H3 predice **negativo**: más vulnerabilidad, recuperación más lenta).

**Tiempo.** Seis ajustes (tres especificaciones por dos asignaturas), de 5 a 10 minutos cada uno. Puedes empezar con `MATERIAS = ["mate"]`.

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")
B = sp.bayes

MATERIAS = ["mate", "lect"]
ITERACIONES = 1000        # calentamiento y muestras por cadena
CADENAS = 2

## 1. Covariables por establecimiento

In [2]:
cov = sp.io_datos.leer_csv(cfg.ruta_salida("dataset_analitico_mate_covariables.csv")).set_index("rbd")
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))

depe = cov["cod_depe1_cat"]
territorio = pd.DataFrame({
    "pobreza_comunal_pre": cov["pobreza_comunal_pre"],
    "log_densidad_comunal": cov["log_densidad_comunal"],
    "log_matricula": np.log(cov["matricula_total_pre"]),
    "rural": cov["cod_rural_rbd_cat"].eq("Rural").astype(float),
    "dep_municipal": depe.isin(["Municipal DAEM", "Municipal Corporación"]).astype(float),
    "dep_slep": depe.eq("Servicio Local de Educación").astype(float),
    "dep_pagado": depe.eq("Particular pagado").astype(float),
}, index=cov.index)

# El IVE de los particulares pagados es 0 por construcción (ausencia estructural);
# su diferencia de nivel la absorbe el indicador dep_pagado.
socio_ive = pd.DataFrame({"ive_pre": cov["ive_pre"]}, index=cov.index)

print("Completitud de las covariables:")
print(pd.concat([territorio, socio_ive], axis=1).notna().mean().round(3).to_string())
print("\nCorrelación entre IVE y GSE ordinal:",
      round(float(cov["ive_pre"].corr(cov["cod_grupo_cat"].map(
          {g: i for i, g in enumerate(B.ORDEN_GSE)}))), 3))

Completitud de las covariables:
pobreza_comunal_pre     0.999
log_densidad_comunal    0.990
log_matricula           0.959
rural                   1.000
dep_municipal           1.000
dep_slep                1.000
dep_pagado              1.000
ive_pre                 0.959

Correlación entre IVE y GSE ordinal: -0.921


**Muestra común.** Para que las tres especificaciones sean comparables, todas se ajustan sobre los mismos establecimientos: los que tienen completas todas las covariables. La especificación A se reajusta aquí sobre esa muestra en lugar de tomarse del notebook 09.

**Nota de la auditoría (24-oct-2026).** El registro dice "2838 de 2838 escuelas con datos completos", pero la muestra efectiva del modelo es de **2.833 escuelas en Matemática y 2.832 en Lectura**: las cinco o seis restantes no tienen puntaje en 2022 y `preparar_datos` las excluye. El informe debe citar 2.833 y 2.832.

In [3]:
completos = pd.concat([territorio, socio_ive], axis=1).dropna().index
print("Establecimientos con todas las covariables:", len(completos))

def panel(materia):
    anios = [2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025]
    L = pd.concat([pd.DataFrame({
        "rbd": ancho["rbd"], "cod_grupo_cat": ancho["cod_grupo_cat"], "anio": t,
        "y": pd.to_numeric(ancho[f"prom_{materia}2m_rbd_{t}"], errors="coerce"),
        "n": pd.to_numeric(ancho[f"nalu_{materia}2m_rbd_{t}"], errors="coerce")})
        for t in anios]).dropna(subset=["y"])
    return L[L["rbd"].isin(completos)]

Establecimientos con todas las covariables: 2960


## 2. Ajuste de las tres especificaciones

In [4]:
especificaciones = {
    "A: GSE + previo": dict(grupo="gse", X=None),
    "B: GSE + previo + territorio": dict(grupo="gse", X=territorio),
    "C: IVE + previo + territorio": dict(grupo="unico", X=pd.concat([socio_ive, territorio], axis=1)),
}
ajustes, datos_esp = {}, {}
for materia in MATERIAS:
    L = panel(materia)
    for nombre, e in especificaciones.items():
        if e["grupo"] == "unico":
            # Sin GSE: un único grupo, y el nivel socioeconómico entra por el IVE.
            Lx = L.assign(grupo="todos")
            datos = B.preparar_datos(Lx, col_grupo="grupo", orden_grupo=("todos",), col_n="n",
                                     covariables_escuela=e["X"])
        else:
            datos = B.preparar_datos(L, col_n="n", covariables_escuela=e["X"])
        m = B.ajustar(datos, tendencia_pre=True, controlar_pre=True,
                      num_warmup=ITERACIONES, num_samples=ITERACIONES, num_chains=CADENAS)
        ajustes[(materia, nombre)] = m
        datos_esp[(materia, nombre)] = datos
        print(materia, "|", nombre, "|", m.diagnosticos)

16:01:27 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


16:01:27 | INFO    | simce_panel.bayes | Datos del modelo: 2833 escuelas, 22392 observaciones.
16:01:30 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.
16:04:22 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 172.0, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 791}
16:04:22 | INFO    | simce_panel.bayes | Covariables: 2838 de 2838 escuelas con datos completos.
16:04:22 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


mate | A: GSE + previo | {'segundos': 172.0, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 791}


16:04:23 | INFO    | simce_panel.bayes | Datos del modelo: 2833 escuelas, 22392 observaciones.
16:09:16 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 293.6, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 852}
16:09:16 | INFO    | simce_panel.bayes | Covariables: 2838 de 2838 escuelas con datos completos.
16:09:16 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


mate | B: GSE + previo + territorio | {'segundos': 293.6, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 852}


...

lect | A: GSE + previo | {'segundos': 173.7, 'divergencias': 0, 'r_hat_max': 1.004, 'n_eff_min': 508}


16:16:36 | INFO    | simce_panel.bayes | Datos del modelo: 2832 escuelas, 22381 observaciones.
16:19:56 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 199.5, 'divergencias': 0, 'r_hat_max': 1.004, 'n_eff_min': 375}
16:19:56 | INFO    | simce_panel.bayes | Covariables: 2838 de 2838 escuelas con datos completos.
16:19:56 | INFO    | simce_panel.bayes | 6 escuelas excluidas por no tener puntaje en 2022.


lect | B: GSE + previo + territorio | {'segundos': 199.5, 'divergencias': 0, 'r_hat_max': 1.004, 'n_eff_min': 375}


16:19:56 | INFO    | simce_panel.bayes | Datos del modelo: 2832 escuelas, 22381 observaciones.
16:23:23 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 207.2, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 433}


lect | C: IVE + previo + territorio | {'segundos': 207.2, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 433}


**Criterio de convergencia:** cero divergencias y R-hat ≤ 1,01 en todos los ajustes. Si alguno no lo cumple, aumenta `ITERACIONES` antes de interpretar.

## 3. H3 en las tres especificaciones

In [5]:
filas = []
for (materia, nombre), m in ajustes.items():
    d = datos_esp[(materia, nombre)]
    if nombre.startswith("C"):
        r = B.resumen_covariables(m, d).set_index("covariable").loc["ive_pre"]
        filas.append({"materia": materia, "especificación": nombre,
                      "medida": "IVE: efecto de +1 DE sobre la pendiente (pts/año)",
                      "estimación": r["recuperacion"], "IC 94 %": r["recuperacion_ic"],
                      "H3 predice": "negativo"})
    else:
        g = B.gradiente_gse(m)
        filas.append({"materia": materia, "especificación": nombre,
                      "medida": "GSE Alto − Bajo en la pendiente (pts/año)",
                      "estimación": g["alto_menos_bajo"], "IC 94 %": g["ic"],
                      "H3 predice": "positivo"})
tabla_h3 = pd.DataFrame(filas)
tabla_h3

,materia,especificación,medida,estimación,IC 94 %,H3 predice
0,mate,A: GSE + previo,GSE Alto − Bajo en la pendiente (pts/año),0.11,"[-1.11, 1.24]",positivo
1,mate,B: GSE + previo + territorio,GSE Alto − Bajo en la pendiente (pts/año),2.39,"[-0.17, 4.81]",positivo
2,mate,C: IVE + previo + territorio,IVE: efecto de +1 DE sobre la pendiente (pts/año),0.49,"[-0.18, 1.14]",negativo
3,lect,A: GSE + previo,GSE Alto − Bajo en la pendiente (pts/año),1.53,"[0.62, 2.4]",positivo
4,lect,B: GSE + previo + territorio,GSE Alto − Bajo en la pendiente (pts/año),1.29,"[-0.9, 3.53]",positivo
5,lect,C: IVE + previo + territorio,IVE: efecto de +1 DE sobre la pendiente (pts/año),-0.19,"[-0.72, 0.34]",negativo


**Cómo leerlo.** La conclusión del notebook 09 (gradiente en Lectura, no en Matemática) es robusta si se repite en las tres filas de cada asignatura: con y sin controles territoriales (A frente a B) y con otra medida del nivel socioeconómico (B frente a C). Si cambia al agregar los controles territoriales, parte del gradiente era territorio; si cambia al pasar del GSE al IVE, la conclusión depende de cómo se mide el nivel socioeconómico, y así debe declararse.

**Advertencia sobre la especificación B.** La dependencia también se superpone con el GSE: casi todos los particulares pagados son de GSE Alto. Al controlar por dependencia, la diferencia Alto − Bajo se estima con los colegios de GSE Alto que **no** son pagados, y pasa a significar "diferencia entre GSE a igual dependencia". Un cambio grande entre A y B con un intervalo mucho más ancho en B es señal de esa superposición, no de un efecto territorial. En ese caso, la comparación más informativa es A frente a C.

## 4. Efectos de las covariables (especificación C)

In [6]:
for materia in MATERIAS:
    clave = (materia, "C: IVE + previo + territorio")
    if clave in ajustes:
        print(f"=== {materia}: efecto de +1 DE de cada covariable (puntos; pendiente en puntos/año)")
        display(B.resumen_covariables(ajustes[clave], datos_esp[clave]))

=== mate: efecto de +1 DE de cada covariable (puntos; pendiente en puntos/año)


,covariable,unidad,caida,caida_ic,caida_creible,recuperacion,recuperacion_ic,recuperacion_creible
0,ive_pre,+1 DE,-1.28,"[-3.16, 0.52]",False,0.49,"[-0.18, 1.14]",False
1,pobreza_comunal_pre,+1 DE,-0.60,"[-1.42, 0.2]",False,0.17,"[-0.14, 0.48]",False
2,log_densidad_comunal,+1 DE,-0.03,"[-0.76, 0.7]",False,0.11,"[-0.17, 0.39]",False
3,log_matricula,+1 DE,-0.55,"[-1.3, 0.19]",False,0.27,"[0.0, 0.54]",True
4,rural,0 → 1,-0.45,"[-3.59, 2.72]",False,0.83,"[-0.39, 2.04]",False
5,dep_municipal,0 → 1,1.70,"[-0.16, 3.45]",False,-0.04,"[-0.71, 0.7]",False
6,dep_slep,0 → 1,2.01,"[-0.23, 4.12]",False,-0.30,"[-1.17, 0.51]",False
7,dep_pagado,0 → 1,22.12,"[18.51, 25.59]",True,1.05,"[-0.3, 2.36]",False


=== lect: efecto de +1 DE de cada covariable (puntos; pendiente en puntos/año)


,covariable,unidad,caida,caida_ic,caida_creible,recuperacion,recuperacion_ic,recuperacion_creible
0,ive_pre,+1 DE,-7.84,"[-9.07, -6.62]",True,-0.19,"[-0.72, 0.34]",False
1,pobreza_comunal_pre,+1 DE,0.01,"[-0.63, 0.66]",False,-0.01,"[-0.27, 0.25]",False
2,log_densidad_comunal,+1 DE,0.57,"[0.01, 1.2]",True,0.26,"[0.03, 0.5]",True
3,log_matricula,+1 DE,-0.93,"[-1.46, -0.38]",True,0.28,"[0.05, 0.5]",True
4,rural,0 → 1,-0.97,"[-3.58, 1.5]",False,0.15,"[-0.89, 1.21]",False
5,dep_municipal,0 → 1,0.63,"[-0.74, 2.06]",False,-0.08,"[-0.61, 0.46]",False
6,dep_slep,0 → 1,1.60,"[-0.06, 3.25]",False,-0.92,"[-1.62, -0.23]",True
7,dep_pagado,0 → 1,1.44,"[-1.05, 4.13]",False,0.38,"[-0.73, 1.55]",False


**Lectura.** Cada coeficiente es condicional a las demás covariables y al nivel previo. Una asociación creíble con la caída pero no con la recuperación indica un factor que pesó en el impacto inicial y no en la trayectoria posterior, o al revés. Son asociaciones, no efectos causales.

**Lectura de esta ejecución (auditoría del 24-oct-2026).**

- `dep_pagado` en la caída de Matemática (+22,1 [18,5; 25,6]): a igual IVE, territorio y **nivel previo**, los particulares pagados cayeron unos 22 puntos menos que la categoría de referencia (particulares subvencionados y administración delegada).
- En Lectura hay más asociaciones creíbles de las que el azar daría (6 de 16): IVE en la caída (−7,8), densidad comunal y matrícula en la caída y en la recuperación, y **`dep_slep` en la recuperación (−0,92 [−1,62; −0,23])**. Esta última no debe omitirse: H5 se contrastó solo en Matemática, de modo que el informe no puede afirmar que el traspaso a SLEP no se asocia a la recuperación en general.
- Son asociaciones pequeñas (0,3 puntos por año por desviación estándar en densidad y matrícula) y condicionales; no son efectos causales.
- El GSE que usa este cuaderno es el más reciente (posterior a 2022). El cuaderno 06, sección 10, muestra la sensibilidad al GSE previo.

In [7]:
tabla_h3.to_csv(cfg.ruta_salida("h3_robustez_covariables.csv"), sep=";", index=False,
                encoding="utf-8-sig")
print("Guardado.")

Guardado.
